# MMM Training GPU Transcription

Run this notebook in Google Colab with a GPU runtime. It transcribes selected Steve Mauro MMM training videos using `faster-whisper` and writes transcript JSON lines under `data/mmm_training/transcripts`.

In [ ]:
!nvidia-smi
!pip install -U faster-whisper ctranslate2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change this if your repo/folder is in a different Drive location.
PROJECT_ROOT = '/content/drive/MyDrive/Helix_V3'
TRAINING_ROOT = f'{PROJECT_ROOT}/data/mmm_training'
SCRIPT_PATH = f'{PROJECT_ROOT}/scripts/colab_transcribe_mmm.py'

# Process one or more IDs. Recommended: start with video_002 while local video_001 runs.
VIDEO_IDS = ['video_002']

# Accuracy/speed tradeoff: base.en is fastest; small.en is usually a good first pass.
MODEL_SIZE = 'small.en'
COMPUTE_TYPE = 'float16'

# Start with 600 seconds for a quick test. Set to None for the full video.
MAX_DURATION_SECONDS = 600

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path(TRAINING_ROOT, 'manifest.json').read_text())
[(item['id'], item['title'], item.get('duration_seconds')) for item in manifest['videos']]

In [ ]:
duration_arg = '' if MAX_DURATION_SECONDS is None else f'--max-duration-seconds {MAX_DURATION_SECONDS}'
for video_id in VIDEO_IDS:
    !python "{SCRIPT_PATH}" --root "{TRAINING_ROOT}" --video-id "{video_id}" --model-size "{MODEL_SIZE}" --device cuda --compute-type "{COMPUTE_TYPE}" --prepare-audio --work-dir /content/mmm_transcribe_work --progress-seconds 300 --srt --overwrite {duration_arg}

In [ ]:
from pathlib import Path
for video_id in VIDEO_IDS:
    path = Path(TRAINING_ROOT, 'transcripts', f'{video_id}.json')
    print(path, path.exists(), path.stat().st_size if path.exists() else 0)
    if path.exists():
        print('\n'.join(path.read_text(encoding='utf-8').splitlines()[:5]))